# Energy Grid OpenEnv — Training Notebook

**Two modes controlled by a single flag:**

| Mode | When | How |
|------|------|-----|
| `LOCAL` | Now, on your machine | Groq API, best-of-G selection, no GPU needed |
| `ONSITE` | Apr 25–26, HuggingFace GPU | Full GRPO, Qwen2.5 fine-tuned with LoRA |

Set `MODE = 'LOCAL'` or `MODE = 'ONSITE'` in cell 2. Everything else is automatic.


## 1. Install Dependencies

In [ ]:
import subprocess, sys

MODE = 'LOCAL'   # set here first so installs are correct

if MODE == 'LOCAL':
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'groq', 'python-dotenv', 'matplotlib', 'numpy', 'pandas'])
else:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'trl>=0.12.0', 'transformers>=4.46.0',
                    'accelerate>=0.34.0', 'peft>=0.13.0',
                    'matplotlib', 'numpy', 'pandas'])
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch',
                    '--index-url', 'https://download.pytorch.org/whl/cu118'])

print(f'Dependencies installed for MODE={MODE}.')

## 2. Mode & Configuration

In [ ]:
# =====================================================================
#  CHANGE THIS FLAG TO SWITCH MODES
#  'LOCAL'  — right now, uses Groq API, no GPU needed
#  'ONSITE' — Apr 25-26, uses HuggingFace GPU + LoRA training
# =====================================================================
MODE = 'LOCAL'
# =====================================================================

assert MODE in ('LOCAL', 'ONSITE')

# Shared
TASK_ID        = 'easy'
G              = 4
MAX_NEW_TOKENS = 128
EVAL_EVERY     = 2
SAVE_EVERY     = 25

if MODE == 'LOCAL':
    GROQ_MODEL   = 'llama-3.3-70b-versatile'
    NUM_EPISODES = 5    # smoke test only
    print(f'LOCAL mode | Model: {GROQ_MODEL} | Episodes: {NUM_EPISODES}')
else:
    HF_MODEL      = 'Qwen/Qwen2.5-0.5B-Instruct'   # T4-safe
    # HF_MODEL    = 'Qwen/Qwen2.5-1.5B-Instruct'   # use if A100
    NUM_EPISODES  = 150
    LEARNING_RATE = 5e-5
    LORA_RANK     = 8
    LORA_ALPHA    = 16
    HF_USERNAME   = 'YOUR_HF_USERNAME'   # replace on the day
    REPO_NAME     = 'energy-grid-grpo-qwen'
    print(f'ONSITE mode | Model: {HF_MODEL} | Episodes: {NUM_EPISODES}')

## 3. Imports

In [ ]:
import os, json, re, time, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import List, Dict, Any, Optional

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# Make repo root importable — open the notebook from your repo root folder
sys.path.insert(0, os.getcwd())

from server.energy_grid_environment import EnergyGridEnvironment
from server.tasks import get_task
from models import EnergyGridAction, EnergyGridObservation

if MODE == 'LOCAL':
    from groq import Groq
    groq_client = Groq(api_key=os.getenv('GROQ_API_KEY'))
    print(f'Groq client ready. Key set: {bool(os.getenv("GROQ_API_KEY"))}')
else:
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    from peft import LoraConfig, get_peft_model, TaskType
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()} | Device: {DEVICE}')
    if torch.cuda.is_available():
        print(f'GPU: {torch.cuda.get_device_name(0)} | '
              f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')

print('Imports done.')

## 4. Load Model

In [ ]:
if MODE == 'LOCAL':
    print('LOCAL mode — Groq API handles inference, no local model needed.')
    model     = None
    tokenizer = None
    optimizer = None
else:
    print(f'Loading {HF_MODEL}...')
    tokenizer = AutoTokenizer.from_pretrained(HF_MODEL, trust_remote_code=True)
    tokenizer.pad_token     = tokenizer.eos_token
    tokenizer.padding_side  = 'left'

    model = AutoModelForCausalLM.from_pretrained(
        HF_MODEL,
        torch_dtype=torch.float16,
        device_map='auto',
        trust_remote_code=True,
    )
    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        target_modules=['q_proj', 'v_proj'],
        lora_dropout=0.05,
        bias='none',
    )
    model     = get_peft_model(model, lora_config)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    model.print_trainable_parameters()
    print('Model ready.')

## 5. Prompt Builder & Action Parser

In [ ]:
SYSTEM_PROMPT = """You are an electricity grid operator. Output ONLY valid JSON:
{"coal_delta":<-100..100>,"hydro_delta":0,"nuclear_delta":0,"battery_mode":"charge|discharge|idle","plant_action":"none","emergency_coal_boost":false,"demand_response_mw":0}

Rules:
- Coal: min 200 MW, max 600 MW, ramp +/-100 MW/step
- Battery: charge/discharge <= 50 MW; keep above 20%
- If demand > supply: increase coal or discharge battery
- If oversupply > 20 MW: reduce coal or charge battery
- Never leave demand unmet if coal headroom exists"""


def obs_to_prompt(obs: EnergyGridObservation) -> str:
    total_gen = obs.coal_mw + obs.solar_mw + obs.wind_mw + obs.hydro_mw + obs.nuclear_mw
    gap       = obs.demand_mw - total_gen
    batt_pct  = int(100 * obs.battery_mwh / max(1, obs.battery_capacity_mwh))
    return (
        f"Step {obs.step}/24 | Demand: {obs.demand_mw:.0f}MW | "
        f"Coal: {obs.coal_mw:.0f}/{obs.coal_max_mw:.0f}MW | "
        f"Battery: {batt_pct}% ({obs.battery_mwh:.0f}MWh) | "
        f"Gap: {gap:+.0f}MW | Freq: {obs.frequency_hz:.2f}Hz | "
        f"Risk: {obs.blackout_risk}"
    )


def parse_action(text: str) -> EnergyGridAction:
    text  = re.sub(r'```[a-z]*\n?|```', '', text).strip()
    match = re.search(r'\{[^{}]*\}', text, re.DOTALL)
    if not match:
        return EnergyGridAction()
    try:
        raw = match.group()
        raw = re.sub(r',\s*}', '}', raw)
        raw = re.sub(r'\bTrue\b',  'true',  raw)
        raw = re.sub(r'\bFalse\b', 'false', raw)
        data = json.loads(raw)
        def clamp(v, lo, hi, d=0.0):
            try:    return max(lo, min(hi, float(v)))
            except: return d
        bm = str(data.get('battery_mode', 'idle')).lower()
        if bm not in ('charge', 'discharge', 'idle'): bm = 'idle'
        return EnergyGridAction(
            coal_delta=clamp(data.get('coal_delta', 0), -100, 100),
            hydro_delta=0.0, nuclear_delta=0.0,
            battery_mode=bm, plant_action='none',
            emergency_coal_boost=bool(data.get('emergency_coal_boost', False)),
            demand_response_mw=clamp(data.get('demand_response_mw', 0), 0, 150),
        )
    except Exception:
        return EnergyGridAction()


print('Prompt builder and parser ready.')

## 6. Sampling Functions

In [ ]:
if MODE == 'LOCAL':

    def _call_groq(system: str, user: str) -> str:
        try:
            resp = groq_client.chat.completions.create(
                model=GROQ_MODEL,
                messages=[{'role': 'system', 'content': system},
                           {'role': 'user',   'content': user}],
                max_tokens=MAX_NEW_TOKENS,
                temperature=0.8,
            )
            return resp.choices[0].message.content or ''
        except Exception as e:
            print(f'[GROQ ERROR] {e}')
            return ''

    def sample_actions(obs, n=G, greedy=False):
        prompt = obs_to_prompt(obs)
        texts  = []
        for _ in range(n):
            texts.append(_call_groq(SYSTEM_PROMPT, prompt))
            time.sleep(1.5)   # stay inside Groq free-tier rate limit
        return texts, [parse_action(t) for t in texts]

    print('LOCAL: Groq sampling ready.')

else:

    def build_chat_prompt(obs):
        return tokenizer.apply_chat_template(
            [{'role': 'system', 'content': SYSTEM_PROMPT},
             {'role': 'user',   'content': obs_to_prompt(obs)}],
            tokenize=False, add_generation_prompt=True
        )

    @torch.no_grad()
    def sample_actions(obs, n=G, greedy=False):
        prompt    = build_chat_prompt(obs)
        inputs    = tokenizer(prompt, return_tensors='pt', padding=True).to(DEVICE)
        input_len = inputs['input_ids'].shape[1]
        outputs   = model.generate(
            input_ids=inputs['input_ids'].repeat(n, 1),
            attention_mask=inputs['attention_mask'].repeat(n, 1),
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=not greedy,
            temperature=0.8 if not greedy else 1.0,
            top_p=0.9 if not greedy else 1.0,
            pad_token_id=tokenizer.eos_token_id,
        )
        texts = [tokenizer.decode(outputs[i][input_len:], skip_special_tokens=True)
                 for i in range(n)]
        return texts, [parse_action(t) for t in texts]

    def compute_log_probs(prompt, completion):
        enc_full   = tokenizer(prompt + completion, return_tensors='pt').to(DEVICE)
        enc_prompt = tokenizer(prompt,              return_tensors='pt').to(DEVICE)
        prompt_len = enc_prompt['input_ids'].shape[1]
        with torch.enable_grad():
            logits = model(**enc_full).logits
        log_probs  = torch.log_softmax(logits[0], dim=-1)
        target_ids = enc_full['input_ids'][0]
        lps = [log_probs[t, target_ids[t+1]]
               for t in range(prompt_len - 1, len(target_ids) - 1)]
        return torch.stack(lps).sum() if lps else torch.tensor(0.0, requires_grad=True, device=DEVICE)

    print('ONSITE: HuggingFace sampling ready.')

## 7. Fast Action Scorer (shared across both modes)

In [ ]:
def _score_action_fast(obs: EnergyGridObservation, action: EnergyGridAction) -> float:
    new_coal = max(0.0, min(obs.coal_max_mw, obs.coal_mw + action.coal_delta))
    batt = 0.0
    if action.battery_mode == 'discharge' and obs.battery_mwh > 5:
        batt = min(50.0, obs.battery_mwh)
    elif action.battery_mode == 'charge':
        batt = -50.0
    gap   = obs.demand_mw - (new_coal + batt)
    score = 0.0
    score -= 0.25  * max(0.0, gap)
    score -= 0.001 * max(0.0, -gap)
    score -= 0.003 * new_coal
    batt_pct = obs.battery_mwh / max(1, obs.battery_capacity_mwh)
    if action.battery_mode == 'discharge' and batt_pct < 0.2:
        score -= 2.0
    if obs.blackout_risk in ('high', 'critical') and action.battery_mode != 'discharge':
        score -= 5.0
    return score

print('Fast scorer ready.')

## 8. Episode Runner

In [ ]:
if MODE == 'LOCAL':

    def run_episode(train=True):
        env  = EnergyGridEnvironment(normalize=False)
        obs  = env.reset(TASK_ID)
        total_reward, step_rewards = 0.0, []
        task = get_task(TASK_ID)
        for step in range(task['total_steps']):
            texts, actions = sample_actions(obs, n=G)
            scores   = [_score_action_fast(obs, a) for a in actions]
            best_idx = int(max(range(G), key=lambda i: scores[i]))
            obs      = env.step(actions[best_idx])
            reward   = obs.reward or 0.0
            total_reward += reward
            step_rewards.append(reward)
            if obs.done: break
        grade = env.get_last_grade() or {}
        return {
            'total_reward': total_reward,
            'score':   grade.get('total_score', 0.0),
            'avg_loss': 0.0,
            'steps':   len(step_rewards),
            'blackout': grade.get('blackout_occurred', False),
        }

    print('LOCAL: episode runner ready.')

else:

    def run_episode(train=True):
        env  = EnergyGridEnvironment(normalize=False)
        obs  = env.reset(TASK_ID)
        total_reward, step_rewards, total_loss = 0.0, [], 0.0
        task = get_task(TASK_ID)
        for step in range(task['total_steps']):
            prompt         = build_chat_prompt(obs)
            texts, actions = sample_actions(obs, n=G, greedy=not train)
            rewards_list   = [_score_action_fast(obs, a) for a in actions]
            rewards_t      = torch.tensor(rewards_list, dtype=torch.float32)
            advantages     = (rewards_t - rewards_t.mean()) / (rewards_t.std() + 1e-8)
            if train:
                loss = torch.tensor(0.0, device=DEVICE, requires_grad=True)
                for i, (text, adv) in enumerate(zip(texts, advantages)):
                    loss = loss + (-adv.to(DEVICE) * compute_log_probs(prompt, text))
                loss = loss / G
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                total_loss += loss.item()
            best_idx = int(rewards_t.argmax())
            obs      = env.step(actions[best_idx])
            reward   = obs.reward or 0.0
            total_reward += reward
            step_rewards.append(reward)
            if obs.done: break
        grade = env.get_last_grade() or {}
        return {
            'total_reward': total_reward,
            'score':    grade.get('total_score', 0.0),
            'avg_loss': total_loss / max(1, task['total_steps']),
            'steps':    len(step_rewards),
            'blackout': grade.get('blackout_occurred', False),
        }

    print('ONSITE: GRPO episode runner ready.')

## 9. Training Loop

In [ ]:
history = {
    'episode': [], 'total_reward': [], 'score': [],
    'avg_loss': [], 'blackout': [],
    'eval_score': [], 'eval_episode': [],
}
os.makedirs('checkpoints', exist_ok=True)

# Baseline eval
print('Running baseline eval...')
if MODE == 'ONSITE': model.eval()
baseline_result = run_episode(train=False)
print(f'Baseline -> score: {baseline_result["score"]:.4f} | '
      f'reward: {baseline_result["total_reward"]:.2f} | '
      f'blackout: {baseline_result["blackout"]}')
history['eval_score'].append(baseline_result['score'])
history['eval_episode'].append(0)

# Main loop
print(f'\nStarting {MODE} run: {NUM_EPISODES} episodes, G={G}...\n')
start_time = time.time()

for episode in range(1, NUM_EPISODES + 1):
    if MODE == 'ONSITE': model.train()
    result = run_episode(train=True)

    history['episode'].append(episode)
    history['total_reward'].append(result['total_reward'])
    history['score'].append(result['score'])
    history['avg_loss'].append(result['avg_loss'])
    history['blackout'].append(int(result['blackout']))

    elapsed = time.time() - start_time
    eta     = elapsed / episode * (NUM_EPISODES - episode)
    print(
        f'Ep {episode:4d}/{NUM_EPISODES} | '
        f'Score: {result["score"]:.4f} | '
        f'Reward: {result["total_reward"]:8.2f} | '
        f'Loss: {result["avg_loss"]:.4f} | '
        f'Blackout: {result["blackout"]} | '
        f'ETA: {eta/60:.1f}min'
    )

    if episode % EVAL_EVERY == 0:
        if MODE == 'ONSITE': model.eval()
        eval_r = run_episode(train=False)
        history['eval_score'].append(eval_r['score'])
        history['eval_episode'].append(episode)
        print(f'  >>> EVAL score: {eval_r["score"]:.4f} | reward: {eval_r["total_reward"]:.2f}')

    if MODE == 'ONSITE' and episode % SAVE_EVERY == 0:
        ckpt = f'checkpoints/ep{episode}'
        model.save_pretrained(ckpt)
        tokenizer.save_pretrained(ckpt)
        pd.DataFrame(history).to_csv('checkpoints/training_history.csv', index=False)
        print(f'  Checkpoint saved: {ckpt}')

print(f'\nDone in {(time.time()-start_time)/60:.1f} minutes.')

## 10. Reward Curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f'Energy Grid OpenEnv — Results ({MODE})', fontsize=14, fontweight='bold')

eps    = history['episode']
window = max(3, len(eps) // 10)

ax = axes[0, 0]
rewards  = history['total_reward']
smoothed = pd.Series(rewards).rolling(window, min_periods=1).mean()
ax.plot(eps, rewards,  alpha=0.3, color='steelblue', label='Raw')
ax.plot(eps, smoothed, color='steelblue', linewidth=2, label=f'MA-{window}')
ax.axhline(y=baseline_result['total_reward'], color='red', linestyle='--', label='Baseline')
ax.set_title('Total Episode Reward')
ax.set_xlabel('Episode'); ax.set_ylabel('Reward')
ax.legend(); ax.grid(alpha=0.3)

ax = axes[0, 1]
scores          = history['score']
smoothed_scores = pd.Series(scores).rolling(window, min_periods=1).mean()
ax.plot(eps, scores,          alpha=0.3, color='green', label='Raw')
ax.plot(eps, smoothed_scores, color='green', linewidth=2, label=f'MA-{window}')
ax.scatter(history['eval_episode'], history['eval_score'],
           color='red', zorder=5, s=60, label='Eval', marker='D')
ax.axhline(y=baseline_result['score'], color='red', linestyle='--', label='Baseline')
ax.set_ylim(0, 1)
ax.set_title('Grader Score (0-1)')
ax.set_xlabel('Episode'); ax.set_ylabel('Score')
ax.legend(); ax.grid(alpha=0.3)

ax = axes[1, 0]
losses        = history['avg_loss']
smoothed_loss = pd.Series(losses).rolling(window, min_periods=1).mean()
ax.plot(eps, losses,        alpha=0.3, color='orange')
ax.plot(eps, smoothed_loss, color='orange', linewidth=2)
ax.set_title('Policy Loss' + (' (N/A in LOCAL mode)' if MODE == 'LOCAL' else ''))
ax.set_xlabel('Episode'); ax.set_ylabel('Loss'); ax.grid(alpha=0.3)

ax = axes[1, 1]
br = pd.Series(history['blackout']).rolling(max(3, len(eps)//5), min_periods=1).mean()
ax.plot(eps, br, color='crimson', linewidth=2)
ax.set_ylim(0, 1)
ax.set_title('Blackout Rate (rolling avg)')
ax.set_xlabel('Episode'); ax.set_ylabel('Fraction'); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: training_curves.png')

## 11. Before vs After

In [ ]:
print('Running final evaluation...')
if MODE == 'ONSITE': model.eval()
final_result = run_episode(train=False)

print('\n' + '='*52)
print(f'  BEFORE vs AFTER  ({MODE}, {TASK_ID} task)')
print('='*52)
print(f'  Metric         Before      After      Delta')
print(f'  {"─"*48}')
b, f = baseline_result, final_result
print(f'  Total Reward  {b["total_reward"]:9.2f}  {f["total_reward"]:9.2f}  {f["total_reward"]-b["total_reward"]:+.2f}')
print(f'  Grader Score  {b["score"]:9.4f}  {f["score"]:9.4f}  {f["score"]-b["score"]:+.4f}')
print(f'  Blackout      {str(b["blackout"]):>9}  {str(f["blackout"]):>9}')
print(f'  Steps         {b["steps"]:9d}  {f["steps"]:9d}')
print('='*52)

delta = f['score'] - b['score']
if delta > 0:
    print(f'\n  Score improved by {delta:.4f} ({delta*100:.1f} percentage points)')
else:
    print(f'\n  No improvement yet — expected for a {NUM_EPISODES}-episode {MODE} smoke test.')
    if MODE == 'LOCAL':
        print('  Switch to MODE=ONSITE on the GPU for real GRPO training.')

## 12. Push to HuggingFace Hub (ONSITE only)

In [ ]:
if MODE == 'LOCAL':
    print('Skipping — no local model to push. Run this cell on the onsite GPU.')
else:
    # from huggingface_hub import login; login()  # uncomment to auth
    print(f'Merging LoRA and pushing to {HF_USERNAME}/{REPO_NAME}...')
    merged = model.merge_and_unload()
    merged.save_pretrained('final_model')
    tokenizer.save_pretrained('final_model')
    merged.push_to_hub(f'{HF_USERNAME}/{REPO_NAME}')
    tokenizer.push_to_hub(f'{HF_USERNAME}/{REPO_NAME}')
    print(f'Done: https://huggingface.co/{HF_USERNAME}/{REPO_NAME}')

## 13. Demo Inference (for the pitch)

In [ ]:
print('Running demo inference...\n')
if MODE == 'ONSITE': model.eval()

demo_env = EnergyGridEnvironment(normalize=False)
obs = demo_env.reset('easy')

for step in range(24):
    texts, actions = sample_actions(obs, n=1, greedy=True)
    action   = actions[0]
    batt_pct = int(100 * obs.battery_mwh / max(1, obs.battery_capacity_mwh))
    gap      = obs.demand_mw - obs.coal_mw
    print(
        f'Step {step+1:02d} | '
        f'Demand={obs.demand_mw:.0f}MW Gap={gap:+.0f}MW | '
        f'Action: coal{action.coal_delta:+.0f} batt={action.battery_mode} | '
        f'Battery={batt_pct}% Freq={obs.frequency_hz:.2f}Hz'
    )
    obs = demo_env.step(action)
    if obs.done: break

grade = demo_env.get_last_grade()
print(f'\nFinal score: {grade["total_score"]:.4f}')
print(f'Components:  {grade["component_scores"]}')